## 第12章「つながりを活かして推薦する」— サポートページ共通セットアップ

サポートページ全体で使用するパスの設定、DataFrame 表示スタイルの適用、および `matplotlib_fontja` のインポートを行う。


In [1]:
import sys
from pathlib import Path

import matplotlib_fontja  # noqa: F401

for path in [Path.cwd(), *Path.cwd().parents]:
    if (path / "common" / "notebook_style.py").exists():
        sys.path.insert(0, str(path))
        break

    if (path / "src" / "common" / "notebook_style.py").exists():
        sys.path.insert(0, str(path / "src"))
        break

    if (path / "doc" / "src" / "common" / "notebook_style.py").exists():
        sys.path.insert(0, str(path / "doc" / "src"))
        break


from common.notebook_style import apply_dataframe_style

apply_dataframe_style()

In [2]:
# CI では短い動作確認に切り替える（通常実行では False）
CI_SMOKE_TEST = False

# 第12章：つながりを活かして推薦する

このファイルでは、第 12 章で扱う BPR、NGCF、LightGCN の実装を `src` から呼び出し、Online Retail の購入履歴を用いた、グラフを使う推薦モデルの学習と評価を実行します。評価時はモデルを 1 回実行して全ユーザーと全アイテムの埋め込みを求め、各バッチの推薦スコアをまとめて計算します。

出典: Daqing Chen, [Online Retail](https://doi.org/10.24432/C5BW33), UCI Machine Learning Repository, 2015。ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)。元データをCSV形式に変換し、本文に示す加工を行います。


## 実行用の準備

次のコードでは、第 12 章の `src/main.py` を探して呼び出せるようにします。`src` を Python がモジュールを探す場所に追加し、作業フォルダを一時的に切り替える補助関数も定義します。


In [2]:
import argparse
import importlib.util
import os
import sys
from contextlib import contextmanager


def find_repo_paths(start: Path | None = None) -> tuple[Path, Path]:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "doc/12/src/main.py").exists():
            return candidate, candidate / "doc/12/src"
        if (candidate / "12/src/main.py").exists():
            return candidate, candidate / "12/src"
    raise FileNotFoundError("第12章の src/main.py が見つかりません。")


@contextmanager
def pushd(path: Path):
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)


REPO_ROOT, SRC_DIR = find_repo_paths()
MAIN_PATH = SRC_DIR / "main.py"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"SRC_DIR   = {SRC_DIR}")


def load_chapter12_main_module():
    spec = importlib.util.spec_from_file_location(
        "chapter12_runtime_main", MAIN_PATH
    )
    if spec is None or spec.loader is None:
        raise ImportError(f"main.py をロードできません: {MAIN_PATH}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def run_chapter12_main(
    model_name: str = "LightGCN", overrides: dict | None = None
):
    module = load_chapter12_main_module()
    args = argparse.Namespace(dataset="online-retail", model=model_name)
    unknown_args_dict = dict(overrides or {})
    unknown_args_dict.setdefault(
        "checkpoint_dir_name", str(SRC_DIR / "checkpoint")
    )
    with pushd(SRC_DIR):
        return module.main(args, unknown_args_dict)

REPO_ROOT = /Users/hiroshi/private/marketing_science/doc
SRC_DIR   = /Users/hiroshi/private/marketing_science/doc/12/src


## 学習設定

次のコードでは、比較するモデル名と共通の設定を指定します。通常は最大 300 エポックとし、10 エポック改善しなければ早期停止します。CI の動作確認では最大 2 エポックです。


In [4]:
# 比較するモデル
MODEL_NAMES = ["BPR", "NGCF", "LightGCN"]

# 実行時に上書きする設定だけを指定する
COMMON_OVERRIDES = {
    "epochs": 2 if CI_SMOKE_TEST else 300,
    "early_stop_num": 2 if CI_SMOKE_TEST else 10,
    "train_batch_size": 1024,
    "eval_batch_size": 128,
    "show_progress": not CI_SMOKE_TEST,
}

print(MODEL_NAMES)
print(COMMON_OVERRIDES)

['BPR', 'NGCF', 'LightGCN']
{'epochs': 300, 'early_stop_num': 10, 'train_batch_size': 1024, 'eval_batch_size': 128, 'show_progress': True}


## 学習の実行

次のコードでは、3 つのモデルを順に学習し、検証データとテストデータの結果をまとめます。


In [5]:
RESULTS = {}
for model_name in MODEL_NAMES:
    overrides = {
        **COMMON_OVERRIDES,
        "result_file_name_postfix": f"notebook_{model_name.lower()}",
    }
    RESULTS[model_name] = run_chapter12_main(model_name, overrides)

RESULTS

2026-09-17T14:53:50 dev INFO [f:main] [L:192] config : {
  "seed": 123,
  "gpu": 0,
  "log_level": "DEBUG",
  "train_val_test": [
    0.8,
    0.1,
    0.1
  ],
  "result_file_name_postfix": "notebook_bpr",
  "checkpoint_dir_name": "/Users/hiroshi/private/marketing_science/doc/12/src/checkpoint",
  "user_id_name": "user_id",
  "item_id_name": "item_id",
  "user_idx_name": "user_idx",
  "item_idx_name": "item_idx",
  "neg_item_idx_name": "neg_item_idx",
  "dataset": "online-retail",
  "usecols": [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID"
  ],
  "positive_rating_threshold": null,
  "model": "BPR",
  "train_batch_size": 1024,
  "data_split": "group_sort",
  "epochs": 300,
  "early_stop_num": 10,
  "eval_batch_size": 128,
  "topk": 10,
  "is_save_data": false,
  "source_format": "online_retail",
  "dataset_lower": 5,
  "dataset_upper": 1000000,
  "split_group_name": "invoice_id",
  "min_user_groups": 3,
  "emb

2026-09-17T14:53:51 dev DEBUG [f:_load_source_dataframe] [L:63] orig_df shape : (541909, 7)


2026-09-17T14:53:51 dev DEBUG [f:_prepare_online_retail_dataframe] [L:114] orig_df shape : (266224, 4)


2026-09-17T14:53:52 dev DEBUG [f:_log_filtered_dataframe_summary] [L:118] user_num   : 1,884


2026-09-17T14:53:52 dev DEBUG [f:_log_filtered_dataframe_summary] [L:119] item_num   : 3,000


2026-09-17T14:53:52 dev DEBUG [f:_log_filtered_dataframe_summary] [L:120] df.head(5) : 
  user_id item_id invoice_id           timestamp
0   17850   21730     536365 2010-12-01 08:26:00
1   17850   22752     536365 2010-12-01 08:26:00
2   17850   71053     536365 2010-12-01 08:26:00
3   17850  84029E     536365 2010-12-01 08:26:00
4   17850  84029G     536365 2010-12-01 08:26:00


... (27 outputs left out)

2026-09-17T15:10:30 dev INFO [f:save_result] [L:177] result file name: 20260917151030_online-retail_LightGCN_notebook_lightgcn.json


2026-09-17T15:10:30 dev INFO [f:save_result] [L:178] successfully saved    : /Users/hiroshi/private/marketing_science/doc/12/src/result/20260917/20260917151030_online-retail_LightGCN_notebook_lightgcn.json


2026-09-17T15:10:30 dev INFO [f:save_result] [L:179] successfully finished : total elapsed_time: 6.87 min


{'BPR': ({'recall@10': 0.05336439229880169,
   'precision@10': 0.0582271771319606,
   'ndcg@10': 0.07424655436457088,
   'mrr@10': 0.15678015585411378,
   'map@10': 0.03444064675875783},
  {'recall@10': 0.034404786689628576,
   'precision@10': 0.036397450281887074,
   'ndcg@10': 0.04575167566252819,
   'mrr@10': 0.09767028578536292,
   'map@10': 0.020012677919345252}),
 'NGCF': ({'recall@10': 0.056183048367753397,
   'precision@10': 0.05806794212122632,
   'ndcg@10': 0.07583651648964851,
   'mrr@10': 0.15684629397787106,
   'map@10': 0.036115383646290765},
  {'recall@10': 0.039140472883372454,
   'precision@10': 0.03783209442965663,
   'ndcg@10': 0.049328622605164685,
   'mrr@10': 0.10329289340060778,
   'map@10': 0.022051882034407156}),
 'LightGCN': ({'recall@10': 0.06030369007410265,
   'precision@10': 0.0593418257504765,
   'ndcg@10': 0.07740983664356726,
   'mrr@10': 0.1605283029519828,
   'map@10': 0.03623858214437076},
  {'recall@10': 0.04031790417641814,
   'precision@10': 0.039

## 実行結果の見方

学習が完了すると、ログに `BEST VALID RESULT` と `TEST RESULT` が表示されます。`BEST VALID RESULT` は各モデルで、検証データの Recall@10 が最も高かった時点の指標で、早期停止と学習途中のモデルを保存する基準になります。`TEST RESULT` は、そのモデルをテストデータで評価した最終的な推薦の精度です。Recall@10 は正解アイテムをどれだけ含められたか、Precision@10 は推薦した上位 10 件のうち正解がどれだけ含まれたか、NDCG@10・MRR@10・MAP@10 は正解アイテムが上位に出ているかを評価する指標です。
